# ToolFailBench — Quickstart

ToolFailBench measures **why** LLM agents fail at tool use, not just whether — across four failure modes: **Tool-Skip**, **Result-Ignore**, **Output-Fabrication**, and **Unnecessary-Tool-Use**.

This notebook runs the benchmark on **one model** over a **small sample**, end to end:
pick a model -> run -> see the four failure-mode rates -> inspect individual traces.

> It uses a closed-API model, so **no GPU is needed**. For the full 1,000-task benchmark and the LLM-judge ensemble, see the README.

## Before you start

From the repo root:
1. `./setup.sh`  (or `pip install -r requirements.txt`)
2. Export the API key for the model you pick — e.g. `XAI_API_KEY` (Grok), `ANTHROPIC_API_KEY` (Claude), `OPENAI_API_KEY` (GPT).

Then run the cells below in order.

In [ ]:
import sys
from pathlib import Path
from collections import Counter
from dotenv import load_dotenv

# Run this notebook from the repo root so the imports below resolve.
sys.path.insert(0, str(Path.cwd()))
load_dotenv()

from models.registry import get_model_config
from evaluation.data import load_tasks_v5
from evaluation.metrics import compute_all_metrics
from scripts.run_eval import load_config, run_single_task

In [ ]:
# Settings -- change these
MODEL      = "grok-4.3"   # any API model in models/configs/ (e.g. claude-haiku-4-5, gpt-5.4-mini-2026-03-17)
DOMAIN     = "finance"    # finance | medical | legal | cybersecurity | real_estate
N_PER_MODE = 3            # tasks per failure mode -- keep small for a fast, cheap demo

In [ ]:
tasks = load_tasks_v5([DOMAIN])

# A small balanced sample: N_PER_MODE tasks of each target failure mode.
by_mode = {}
for t in tasks:
    by_mode.setdefault(t["target_failure_mode"], []).append(t)
sample = [t for mode in by_mode for t in by_mode[mode][:N_PER_MODE]]

print(f"{len(sample)} tasks from '{DOMAIN}':", {m: min(N_PER_MODE, len(v)) for m, v in by_mode.items()})

ex = sample[0]
print("\nExample task")
print("  user asks    :", ex["user_message"])
print("  tool returns :", ex["mock_tool_return"])
print("  must contain :", ex["ground_truth"]["answer_must_contain"])

## Run the sample

Each task is a **two-call exchange**: the model sees the question and the available tools, we inject the tool's return value (deliberately chosen to contradict what the model likely memorized), then ask for a final answer. The rule classifier then labels the trace.

In [ ]:
config = load_config()
model_cfg = get_model_config(MODEL)

results = []
for i, t in enumerate(sample, 1):
    results.append(run_single_task(t, model_cfg, config))
    print(f"\r  ran {i}/{len(sample)} tasks", end="", flush=True)
print("  done")

In [ ]:
m = compute_all_metrics(results)
print(f"Clean Tool-Use Rate (CTUR):  {m['ctur']:.0%}")
print(f"Tool-Skip Rate (TSR):        {m['tsr']:.0%}")
print(f"Result-Ignore Rate (RIR):    {m['rir']:.0%}")
print(f"Output-Fabrication (OFR):    {m['ofr']:.0%}")
print(f"Unnecessary-Tool Use (UTR):  {m['utr']:.0%}")
print("\nlabels:", dict(Counter(r["classification"] for r in results)))

## Inspect the traces

A label is a property of *what the model did*. Below: the classifier's label, the task's intended mode, whether a tool was actually called, and the model's answer.

*(Rates above are over a small sample — illustrative, not the benchmark numbers. The full run is 1,000 tasks.)*

In [ ]:
for r in results:
    called = "tool called" if r["agent_trace"]["tool_calls"] else "no tool call"
    print(f"[{r['classification']:18s}]  target={r['task']['target_failure_mode']:16s}  {called}")
    print(f"   -> {(r['agent_answer'] or '').strip()[:160]}\n")

## Next steps

- **Full benchmark** (1,000 tasks): `make eval MODEL=grok-4.3`  — or  `python scripts/run_eval.py --model grok-4.3`
- **Dual-judge ensemble**: `make judge EVAL=results/v5/<your-file>.json`
- **Reproduce the paper leaderboard**: download the result traces into `results/v5/`, then `python evaluation/validate_results.py`

See the **README** for the full guide.